# Sensibilites avec recalibration

Ce notebook reprend volontairement les memes definitions et le meme style que le notebook `code_deep_hedging.ipynb` present sur `main` : fonctions Black--Scholes, simulation de l'actif, reseau `NeuralNetwork`, PnL actualise, couts proportionnels.

La difference principale est la suivante : au lieu d'entrainer une seule strategie sur un scenario de reference puis de la tester sur des scenarios differents, on **recalibre** le modele pour chaque scenario. Autrement dit, pour chaque volatilite ou strike teste, on re-simule les donnees, on re-entraine le reseau, puis on evalue hors-echantillon sur le meme scenario.

Cela mesure la robustesse de la methode Deep Hedging quand le modele est recalibre au scenario courant.

In [ ]:
# Partie fonctions essentielles -- reprise du style du notebook main

import math
import numpy as np
from scipy import stats
from scipy.stats import norm
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn

try:
    import seaborn as sns
    sns.set_theme()
except Exception:
    pass

np.random.seed(42)
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")


def monte_carlo(sample, proba=0.95):
    mean = np.mean(sample).item()
    var = np.var(sample, ddof=1).item()
    alpha = 1 - proba
    quantile = stats.norm.ppf(1 - alpha / 2).item()
    ci_size = quantile * math.sqrt(var / sample.size)
    return {"mean": mean, "var": var, "lower": mean - ci_size, "upper": mean + ci_size}


def d1(spot, t, r, sigma, strike):
    t = np.maximum(t, 1e-12)
    return (np.log(spot / strike) + t * (r + 0.5 * sigma**2)) / (sigma * np.sqrt(t))


def d2(spot, t, r, sigma, strike):
    return d1(spot, t, r, sigma, strike) - sigma * np.sqrt(t)


def price_call_BS(spot, t, r, sigma, strike):
    d1_ = d1(spot, t, r, sigma, strike)
    d2_ = d2(spot, t, r, sigma, strike)
    return spot * norm.cdf(d1_) - strike * np.exp(-r * t) * norm.cdf(d2_)


def price_put_BS(spot, t, r, sigma, strike):
    return price_call_BS(spot, t, r, sigma, strike) - spot + strike * np.exp(-r * t)


def dprice_put_BS(spot, t, r, sigma, strike):
    return norm.cdf(d1(spot, t, r, sigma, strike)) - 1


class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, x):
        return self.linear_relu_stack(x)


def simulation_brownien(n, T, M):
    B = np.zeros((M, n + 1))
    B[:, 1:] = np.cumsum(np.sqrt(T / n) * np.random.standard_normal(size=(M, n)), axis=1)
    return B


def simulation_actif(n, T, M, S0, sigma, r):
    S = np.zeros((M, n + 1))
    S[:, 0] = S0
    brownian = simulation_brownien(n, T, M)
    dt = T / n
    for i in range(1, n + 1):
        S[:, i] = S[:, i - 1] * np.exp(
            (r - 0.5 * sigma**2) * dt + sigma * (brownian[:, i] - brownian[:, i - 1])
        )
    return S


def payoff_put(S, K):
    return np.maximum(0, K - S)

## Parametres communs

On garde les memes conventions que dans le notebook initial : PnL actualise, reseau MLP, entree avec memoire `delta_prev`, cout proportionnel de transaction.

Pour que le notebook reste executable facilement, les tailles Monte Carlo sont volontairement plus petites que dans le notebook `main`. On peut augmenter `M` et `M_test` pour obtenir des resultats plus stables.

In [ ]:
# Parametres globaux
r = 0.1
S0 = 100.0
N, T = 10, 1.0

H = 32
input_size = 3          # S normalise, tau, delta_{t-1}
output_size = 1
learning_rate = 1e-3
n_epochs = 6
batch_size = 1000
M = 50_000              # augmenter vers 10**6 pour une experience finale
M_test = 10_000
eps = 5e-2              # cout proportionnel, comme dans le notebook main

print({
    "N": N,
    "T": T,
    "H": H,
    "M_train": M,
    "M_test": M_test,
    "eps": eps,
})

In [ ]:
def prepare_data(S0, K, sigma, r, N, T, M):
    """Prepare les tenseurs d'entrainement avec les memes conventions que main."""
    sample_S_raw = simulation_actif(N, T, M, S0, sigma, r)
    times = np.arange(N + 1) * T / N

    moyenne = S0 * np.exp(r * times)
    ecart_type = S0 * np.exp(r * times) * np.sqrt(np.exp(sigma**2 * times) - 1)
    ecart_type[0] = 1.0

    sample_S_norm = (sample_S_raw - moyenne) / ecart_type
    S_act = sample_S_raw * np.exp(-r * times)
    dS_np = np.diff(S_act, axis=1)

    tau_grid = (T - times[:-1]) / T
    data_np = np.stack([
        sample_S_norm[:, :-1],
        np.repeat(tau_grid.reshape(1, -1), M, axis=0),
    ], axis=2)

    phi_np = payoff_put(sample_S_raw[:, -1], K) * np.exp(-r * T)

    return {
        "sample_S_raw": sample_S_raw,
        "sample_S_norm": sample_S_norm,
        "S_act": S_act,
        "dS": dS_np,
        "data": data_np,
        "phi": phi_np,
        "times": times,
        "moyenne": moyenne,
        "ecart_type": ecart_type,
        "tau_grid": tau_grid,
        "P0_val": float(price_put_BS(S0, T, r, sigma, K)),
    }


def train_recalibrated_model(K, sigma, label):
    """Entraine un modele pour un scenario donne : recalibration complete."""
    np.random.seed(42)
    torch.manual_seed(42)

    pack = prepare_data(S0, K, sigma, r, N, T, M)
    data = torch.from_numpy(pack["data"]).float().to(device)
    dS_torch = torch.from_numpy(pack["dS"]).float().to(device)
    phi = torch.from_numpy(pack["phi"]).float().to(device)
    S_act_torch = torch.from_numpy(pack["S_act"]).float().to(device)
    P0 = torch.tensor(pack["P0_val"], dtype=torch.float32, device=device)

    model = NeuralNetwork(input_size=input_size, hidden_size=H, output_size=output_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    size_tot = data.shape[0]
    nb_batch = size_tot // batch_size
    losses = []

    for epoch in range(n_epochs):
        perm = torch.randperm(size_tot, device=device)
        epoch_loss = 0.0

        for j in range(nb_batch):
            idx = perm[j * batch_size:(j + 1) * batch_size]
            X_batch = data[idx]
            dS_batch = dS_torch[idx]
            phi_batch = phi[idx]
            S_batch = S_act_torch[idx]
            batch_n = X_batch.shape[0]

            delta_prev = torch.zeros(batch_n, device=device)
            q_list = []

            for t in range(N):
                S_t = X_batch[:, t, 0]
                tau_t = X_batch[:, t, 1]
                input_t = torch.stack([S_t, tau_t, delta_prev], dim=1)
                delta_t = model(input_t).squeeze(-1)
                q_list.append(delta_t)
                delta_prev = delta_t

            q = torch.stack(q_list, dim=1)
            delta_pnl = (q * dS_batch).sum(dim=1)

            q_prev = torch.cat([torch.zeros(batch_n, 1, device=device), q], dim=1)
            q_next = torch.cat([q, torch.zeros(batch_n, 1, device=device)], dim=1)
            costs = eps * torch.sum(S_batch * torch.abs(q_next - q_prev), dim=1)

            pnl = P0 - phi_batch + delta_pnl - costs
            loss = torch.mean(pnl**2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        epoch_loss /= nb_batch
        losses.append(epoch_loss)
        print(f"{label} | Epoch {epoch + 1}/{n_epochs} | Loss = {epoch_loss:.6f}")

    return model, losses, pack

In [ ]:
def evaluate_recalibrated_model(model, K, sigma):
    """Evalue hors-echantillon le modele recalibre sur le meme scenario."""
    np.random.seed(123)

    sample_S_test = simulation_actif(N, T, M_test, S0, sigma, r)
    times = np.arange(N + 1) * T / N

    moyenne = S0 * np.exp(r * times)
    ecart_type = S0 * np.exp(r * times) * np.sqrt(np.exp(sigma**2 * times) - 1)
    ecart_type[0] = 1.0

    sample_norm_test = (sample_S_test - moyenne) / ecart_type
    tau_grid = (T - np.arange(N) * T / N) / T
    tau_mat = np.repeat(tau_grid.reshape(1, -1), M_test, axis=0)

    sample_S_test_disc = sample_S_test * np.exp(-r * times)
    dS_test = np.diff(sample_S_test_disc, axis=1)
    payoff_disc = np.exp(-r * T) * np.maximum(K - sample_S_test[:, -1], 0)
    P0_val = float(price_put_BS(S0, T, r, sigma, K))

    # Evaluation du reseau avec memoire delta_{t-1}
    model.eval()
    delta_prev = np.zeros(M_test)
    q_list = []
    with torch.no_grad():
        for t in range(N):
            S_t = sample_norm_test[:, t]
            tau_t = tau_mat[:, t]
            X_input = np.stack([S_t, tau_t, delta_prev], axis=1)
            X_input = torch.from_numpy(X_input).float().to(device)
            delta_t = model(X_input).squeeze(-1).cpu().numpy()
            q_list.append(delta_t)
            delta_prev = delta_t

    q_nn = np.stack(q_list, axis=1)
    q_prev_nn = np.concatenate([np.zeros((M_test, 1)), q_nn], axis=1)
    q_next_nn = np.concatenate([q_nn, np.zeros((M_test, 1))], axis=1)
    costs_nn = eps * np.sum(sample_S_test_disc * np.abs(q_next_nn - q_prev_nn), axis=1)
    delta_pnl_nn = np.sum(q_nn * dS_test, axis=1)
    pnl_nn = P0_val - payoff_disc + delta_pnl_nn - costs_nn

    # Benchmark delta BS sur le meme scenario
    delta_bs = np.zeros((N, M_test))
    sample_bs = sample_S_test.T
    for n in range(N):
        tau_n = T - n * T / N
        delta_bs[n] = dprice_put_BS(sample_bs[n], tau_n, r, sigma, K)
    q_bs = delta_bs.T

    q_prev_bs = np.concatenate([np.zeros((M_test, 1)), q_bs], axis=1)
    q_next_bs = np.concatenate([q_bs, np.zeros((M_test, 1))], axis=1)
    costs_bs = eps * np.sum(sample_S_test_disc * np.abs(q_next_bs - q_prev_bs), axis=1)
    delta_pnl_bs = np.sum(q_bs * dS_test, axis=1)
    pnl_bs = P0_val - payoff_disc + delta_pnl_bs - costs_bs

    return {
        "pnl_nn": pnl_nn,
        "pnl_bs": pnl_bs,
        "costs_nn": costs_nn,
        "costs_bs": costs_bs,
        "q_nn": q_nn,
        "q_bs": q_bs,
    }


def cvar_left(pnl, alpha=0.95):
    threshold = np.quantile(pnl, 1 - alpha)
    return pnl[pnl <= threshold].mean()


def metrics_from_pnl(name, pnl, costs):
    mc = monte_carlo(pnl)
    return {
        "Model": name,
        "Mean": np.mean(pnl),
        "Variance": np.var(pnl),
        "Std": np.std(pnl),
        "MSE": np.mean(pnl**2),
        "Q01": np.quantile(pnl, 0.01),
        "Q05": np.quantile(pnl, 0.05),
        "Q50": np.quantile(pnl, 0.50),
        "Q95": np.quantile(pnl, 0.95),
        "Q99": np.quantile(pnl, 0.99),
        "CVaR_95_left": cvar_left(pnl, 0.95),
        "Mean_cost": np.mean(costs),
        "CI_mean_low": mc["lower"],
        "CI_mean_high": mc["upper"],
    }

## Sensibilite a la volatilite avec recalibration

Dans le notebook `main`, le modele etait entraine sur une volatilite de reference puis teste sur plusieurs volatilites. Ici, on fait une recalibration complete : pour chaque `sigma`, on re-entraine le reseau avec cette meme volatilite, puis on teste hors-echantillon avec cette volatilite.

In [ ]:
sigma_range = [0.10, 0.20, 0.30]
K_ref = 100

results_sigma = []
pnl_by_sigma = {}
losses_by_sigma = {}

for sigma_scenario in sigma_range:
    label = f"sigma={sigma_scenario:.2f}"
    print("\n" + "=" * 70)
    print(f"Recalibration pour {label}")
    print("=" * 70)

    model, losses, _ = train_recalibrated_model(K=K_ref, sigma=sigma_scenario, label=label)
    eval_pack = evaluate_recalibrated_model(model, K=K_ref, sigma=sigma_scenario)
    losses_by_sigma[label] = losses
    pnl_by_sigma[label] = eval_pack

    row_nn = metrics_from_pnl("NN recalibre", eval_pack["pnl_nn"], eval_pack["costs_nn"])
    row_bs = metrics_from_pnl("Delta BS", eval_pack["pnl_bs"], eval_pack["costs_bs"])
    row_nn["Scenario"] = label
    row_bs["Scenario"] = label
    results_sigma.extend([row_nn, row_bs])

sigma_table = pd.DataFrame(results_sigma)
sigma_table[["Scenario", "Model", "Mean", "Std", "MSE", "Q01", "Q99", "CVaR_95_left", "Mean_cost"]].round(6)

In [ ]:
fig, axes = plt.subplots(len(sigma_range), 1, figsize=(8, 4 * len(sigma_range)))
if len(sigma_range) == 1:
    axes = [axes]

for ax, sigma_scenario in zip(axes, sigma_range):
    label = f"sigma={sigma_scenario:.2f}"
    pnl_nn = pnl_by_sigma[label]["pnl_nn"]
    pnl_bs = pnl_by_sigma[label]["pnl_bs"]
    all_pnl = np.concatenate([pnl_nn, pnl_bs])
    bins = np.linspace(np.quantile(all_pnl, 0.001), np.quantile(all_pnl, 0.999), 60)
    ax.hist(pnl_nn, bins=bins, density=True, alpha=0.55, label="NN recalibre")
    ax.hist(pnl_bs, bins=bins, density=True, alpha=0.35, label="Delta BS")
    ax.axvline(0, linestyle="--", color="black", linewidth=1)
    ax.set_title(rf"Recalibration pour $\sigma = {sigma_scenario}$")
    ax.set_ylabel("Densite")
    ax.legend()

axes[-1].set_xlabel("PnL final actualise")
plt.tight_layout()
plt.show()

## Sensibilite au strike avec recalibration

On applique la meme logique au strike. Pour chaque valeur de `K`, on recalcule la prime Black--Scholes, on simule les donnees avec ce payoff, on re-entraine le reseau, puis on evalue sur un test set independant avec le meme `K`.

In [ ]:
K_range = [80, 100, 120]
sigma_ref = 0.20

results_K = []
pnl_by_K = {}
losses_by_K = {}

for K_scenario in K_range:
    label = f"K={K_scenario}"
    print("\n" + "=" * 70)
    print(f"Recalibration pour {label}")
    print("=" * 70)

    model, losses, _ = train_recalibrated_model(K=K_scenario, sigma=sigma_ref, label=label)
    eval_pack = evaluate_recalibrated_model(model, K=K_scenario, sigma=sigma_ref)
    losses_by_K[label] = losses
    pnl_by_K[label] = eval_pack

    row_nn = metrics_from_pnl("NN recalibre", eval_pack["pnl_nn"], eval_pack["costs_nn"])
    row_bs = metrics_from_pnl("Delta BS", eval_pack["pnl_bs"], eval_pack["costs_bs"])
    row_nn["Scenario"] = label
    row_bs["Scenario"] = label
    results_K.extend([row_nn, row_bs])

K_table = pd.DataFrame(results_K)
K_table[["Scenario", "Model", "Mean", "Std", "MSE", "Q01", "Q99", "CVaR_95_left", "Mean_cost"]].round(6)

In [ ]:
fig, axes = plt.subplots(len(K_range), 1, figsize=(8, 4 * len(K_range)))
if len(K_range) == 1:
    axes = [axes]

for ax, K_scenario in zip(axes, K_range):
    label = f"K={K_scenario}"
    pnl_nn = pnl_by_K[label]["pnl_nn"]
    pnl_bs = pnl_by_K[label]["pnl_bs"]
    all_pnl = np.concatenate([pnl_nn, pnl_bs])
    bins = np.linspace(np.quantile(all_pnl, 0.001), np.quantile(all_pnl, 0.999), 60)
    ax.hist(pnl_nn, bins=bins, density=True, alpha=0.55, label="NN recalibre")
    ax.hist(pnl_bs, bins=bins, density=True, alpha=0.35, label="Delta BS")
    ax.axvline(0, linestyle="--", color="black", linewidth=1)
    ax.set_title(rf"Recalibration pour $K = {K_scenario}$")
    ax.set_ylabel("Densite")
    ax.legend()

axes[-1].set_xlabel("PnL final actualise")
plt.tight_layout()
plt.show()

## Lecture des resultats

Cette version repond a une question differente du notebook `main` :

- notebook `main` : **generalisation hors distribution** ; un modele entraine sur un scenario est teste sur un autre ;
- notebook present : **robustesse par recalibration** ; pour chaque scenario, le modele est re-entraine puis teste dans ce meme scenario.

La recalibration doit normalement produire des resultats plus stables que le test hors distribution. Si la performance se degrade malgre la recalibration, cela indique que le scenario est intrinsequement plus difficile : option plus convexe, volatilite plus elevee, couts plus importants ou strike plus expose.